# 02 · Embedding and Chroma Indexing
Covers embedding creation, batch vector indexing in Chroma, idempotency, document active toggles, and vector deletion.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## 1. Generate Embeddings

In [ ]:
from app.providers import get_embeddings
import synthetic
from app.parsers import parse_bytes
from app.chunker import chunk_sections
files = synthetic.make_all()
secs = parse_bytes(files["BIO101"]["bio_notes.pdf"], "bio_notes.pdf")
chunks = chunk_sections(secs, "demo-doc", {"subject_code":"BIO101","is_active":True,"file_name":"bio_notes.pdf"})
vecs = get_embeddings().embed_documents([c.page_content for c in chunks])
print("Vectors generated:", len(vecs), "x", len(vecs[0]), "dimensions")

## 2. Ingest the Complete Synthetic Corpus into Chroma

In [ ]:
from app.ingest import ingest_document, get_status
results = {}
for subj, fs in files.items():
    for name, data in fs.items():
        did = f"{subj}-{name}"
        results[did] = ingest_document(did, {"subject": subj, "is_active": True}, data=data, file_name=name)
import pandas as pd
pd.DataFrame(results).T[["chunkCount","durationMs","collection"]]

## 3. Verify Vector Store Ingestion Status

In [ ]:
from app.store import get_store
st = get_store()
print("Total chunks in vector store:", st.count())
print("Ingestion status for BIO101-bio_notes.pdf:", get_status("BIO101-bio_notes.pdf"))

## 4. Idempotent Ingestion & Deletion Testing

In [ ]:
n0 = len(st.chunks_of("PHY101-physics.docx"))
ingest_document("PHY101-physics.docx", {"subject":"PHY101","is_active":True}, data=files["PHY101"]["physics.docx"], file_name="physics.docx")
assert len(st.chunks_of("PHY101-physics.docx")) == n0, "Re-ingestion must be idempotent without duplicates"
print("Idempotency verified: chunk count is stable at", n0)